In [ ]:
from google.colab import files

uploaded = files.upload()

Saving train.csv to train.csv


In [ ]:
import pandas as pd

df = pd.read_csv("train.csv")

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (13389, 2)
     Category                                               Text
0  Accountant  education omba executive leadership university...
1  Accountant  howard gerrard accountant deyjobcom birmingham...
2  Accountant  kevin frank senior accountant inforesumekraftc...
3  Accountant  place birth nationality olivia ogilvy accounta...
4  Accountant  stephen greet cpa senior accountant 9 year exp...


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# =========================================================
# 2. REMOVE EXACT DUPLICATES
# =========================================================

df = df.drop_duplicates().reset_index(drop=True)

print("After removing exact duplicates:", df.shape)


# =========================================================
# 3. REMOVE CONFLICTING RESUMES
#    Same resume text having different career labels
# =========================================================

text_label_count = df.groupby("Text")["Category"].nunique()

conflicting_texts = text_label_count[
    text_label_count > 1
].index

print("Conflicting resume texts:", len(conflicting_texts))

df = df[~df["Text"].isin(conflicting_texts)].copy()
df = df.reset_index(drop=True)

print("Final dataset:", df.shape)


# =========================================================
# 4. CHECK CATEGORY DISTRIBUTION
# =========================================================

print("\nNumber of categories:", df["Category"].nunique())

print("\nCategory distribution:")
print(df["Category"].value_counts())


# =========================================================
# 5. INPUT AND TARGET
# =========================================================

X = df["Text"].astype(str)
y = df["Category"].astype(str)


# =========================================================
# 6. TRAIN / TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))


# =========================================================
# 7. TF-IDF
# =========================================================

tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_features=15000,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("\nTF-IDF training shape:", X_train_tfidf.shape)
print("TF-IDF testing shape:", X_test_tfidf.shape)


# =========================================================
# 8. RANDOM FOREST
# =========================================================

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("\nTraining Random Forest...")

rf.fit(X_train_tfidf, y_train)

print("Training completed.")


# =========================================================
# 9. EVALUATION
# =========================================================

train_predictions = rf.predict(X_train_tfidf)
test_predictions = rf.predict(X_test_tfidf)

train_accuracy = accuracy_score(
    y_train,
    train_predictions
)

test_accuracy = accuracy_score(
    y_test,
    test_predictions
)

accuracy_gap = train_accuracy - test_accuracy

print("\n==============================")
print("RANDOM FOREST RESULTS")
print("==============================")

print(f"Training Accuracy : {train_accuracy * 100:.2f}%")
print(f"Testing Accuracy  : {test_accuracy * 100:.2f}%")
print(f"Accuracy Gap      : {accuracy_gap * 100:.2f}%")

After removing exact duplicates: (12275, 2)
Conflicting resume texts: 169
Final dataset: (11916, 2)

Number of categories: 43

Category distribution:
Category
Education                    388
Electrical Engineering       360
Consultant                   343
Sales                        342
Digital Media                338
Accountant                   336
Mechanical Engineer          335
Building and Construction    333
Finance                      330
Aviation                     327
Operations Manager           327
Testing                      322
Management                   316
Apparel                      311
Business Analyst             310
Public Relations             309
Civil Engineer               307
Network Security Engineer    306
Human Resources              303
Architecture                 302
Automobile                   300
Health and Fitness           295
Banking                      290
Java Developer               288
SAP Developer                283
Advocate        

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Starting 5-Fold Cross-Validation...")

cv_scores = cross_val_score(
    rf,
    X_train_tfidf,
    y_train,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

print("\n5-Fold Cross-Validation Results")
print("--------------------------------")

for i, score in enumerate(cv_scores, 1):
    print(f"Fold {i}: {score * 100:.2f}%")

print(f"\nMean CV Accuracy: {cv_scores.mean() * 100:.2f}%")
print(f"CV Standard Deviation: {cv_scores.std() * 100:.2f}%")

Starting 5-Fold Cross-Validation...

5-Fold Cross-Validation Results
--------------------------------
Fold 1: 79.60%
Fold 2: 79.34%
Fold 3: 79.64%
Fold 4: 77.33%
Fold 5: 80.22%

Mean CV Accuracy: 79.23%
CV Standard Deviation: 0.99%


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [200, 300],
    "max_depth": [15, 20, 25],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt"]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

print("Starting Random Forest hyperparameter tuning...")

grid_search.fit(X_train_tfidf, y_train)

print("\nBest Parameters:")
print(grid_search.best_params_)

print("\nBest 5-Fold CV Accuracy:")
print(f"{grid_search.best_score_ * 100:.2f}%")

Starting Random Forest hyperparameter tuning...
Fitting 5 folds for each of 24 candidates, totalling 120 fits

Best Parameters:
{'max_depth': 25, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 300}

Best 5-Fold CV Accuracy:
80.25%


In [ ]:
best_rf = grid_search.best_estimator_

tuned_train_pred = best_rf.predict(X_train_tfidf)
tuned_test_pred = best_rf.predict(X_test_tfidf)

tuned_train_accuracy = accuracy_score(
    y_train,
    tuned_train_pred
)

tuned_test_accuracy = accuracy_score(
    y_test,
    tuned_test_pred
)

print("\n==============================")
print("TUNED RANDOM FOREST RESULTS")
print("==============================")

print(f"Training Accuracy : {tuned_train_accuracy * 100:.2f}%")
print(f"Testing Accuracy  : {tuned_test_accuracy * 100:.2f}%")
print(
    f"Accuracy Gap      : "
    f"{(tuned_train_accuracy - tuned_test_accuracy) * 100:.2f}%"
)


TUNED RANDOM FOREST RESULTS
Training Accuracy : 97.56%
Testing Accuracy  : 80.96%
Accuracy Gap      : 16.60%


In [ ]:
!pip install -q xgboost

In [ ]:
from xgboost import XGBClassifier

In [ ]:
!pip install -q xgboost
from xgboost import XGBClassifier

print("XGBoost ready.")

XGBoost ready.


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

# Fit encoder only on training labels
y_train_encoded = label_encoder.fit_transform(y_train)

# Transform validation/test labels using the same encoder
y_test_encoded = label_encoder.transform(y_test)

print("Number of classes:", len(label_encoder.classes_))
print("First 10 classes:")
print(label_encoder.classes_[:10])

Number of classes: 43
First 10 classes:
['Accountant' 'Advocate' 'Agriculture' 'Apparel' 'Architecture' 'Arts'
 'Automobile' 'Aviation' 'BPO' 'Banking']


In [ ]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

print("Training XGBoost...")

xgb.fit(
    X_train_tfidf,
    y_train_encoded
)

print("XGBoost training completed.")

Training XGBoost...
XGBoost training completed.


In [ ]:
xgb_train_pred_encoded = xgb.predict(X_train_tfidf)
xgb_test_pred_encoded = xgb.predict(X_test_tfidf)

# Convert predictions back to original category names
xgb_train_pred = label_encoder.inverse_transform(
    xgb_train_pred_encoded.astype(int)
)

xgb_test_pred = label_encoder.inverse_transform(
    xgb_test_pred_encoded.astype(int)
)

In [ ]:
xgb_train_accuracy = accuracy_score(
    y_train,
    xgb_train_pred
)

xgb_test_accuracy = accuracy_score(
    y_test,
    xgb_test_pred
)

print("\n==============================")
print("XGBOOST BASELINE RESULTS")
print("==============================")

print(f"Training Accuracy : {xgb_train_accuracy * 100:.2f}%")
print(f"Testing Accuracy  : {xgb_test_accuracy * 100:.2f}%")
print(
    f"Accuracy Gap      : "
    f"{(xgb_train_accuracy - xgb_test_accuracy) * 100:.2f}%"
)


XGBOOST BASELINE RESULTS
Training Accuracy : 99.97%
Testing Accuracy  : 85.32%
Accuracy Gap      : 14.65%


In [ ]:
from sklearn.model_selection import GridSearchCV

xgb_param_grid = {
    "n_estimators": [100],
    "max_depth": [4, 6],
    "learning_rate": [0.1],
    "subsample": [0.8],
    "colsample_bytree": [0.8]
}

xgb_base = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

xgb_grid = GridSearchCV(
    estimator=xgb_base,
    param_grid=xgb_param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

print("Starting FAST XGBoost tuning...")

xgb_grid.fit(
    X_train_tfidf,
    y_train_encoded
)

print("\nBest XGBoost Parameters:")
print(xgb_grid.best_params_)

print("\nBest 3-Fold CV Accuracy:")
print(f"{xgb_grid.best_score_ * 100:.2f}%")

Starting FAST XGBoost tuning...
Fitting 3 folds for each of 2 candidates, totalling 6 fits

Best XGBoost Parameters:
{'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 100, 'subsample': 0.8}

Best 3-Fold CV Accuracy:
84.71%


In [ ]:
best_xgb = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

print("Training final XGBoost...")

best_xgb.fit(
    X_train_tfidf,
    y_train_encoded
)

print("Completed.")

Training final XGBoost...
Completed.


In [ ]:
pred_test_encoded = best_xgb.predict(X_test_tfidf)

pred_test = label_encoder.inverse_transform(
    pred_test_encoded.astype(int)
)

final_xgb_accuracy = accuracy_score(
    y_test,
    pred_test
)

print(f"Final Tuned XGBoost Test Accuracy: {final_xgb_accuracy * 100:.2f}%")

Final Tuned XGBoost Test Accuracy: 85.36%


In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [ ]:
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Sentence-BERT model loaded.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence-BERT model loaded.


In [ ]:
print("Creating training embeddings...")

train_embeddings = sbert_model.encode(
    X_train.tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Training embeddings shape:", train_embeddings.shape)

Creating training embeddings...


Batches:   0%|          | 0/149 [00:00<?, ?it/s]

Training embeddings shape: (9532, 384)


In [ ]:
print("Creating test embeddings...")

test_embeddings = sbert_model.encode(
    X_test.tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Test embeddings shape:", test_embeddings.shape)

Creating test embeddings...


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Test embeddings shape: (2384, 384)


In [ ]:
categories = sorted(y_train.unique())

career_embeddings = {}

for category in categories:
    category_mask = (y_train.values == category)
    career_embeddings[category] = train_embeddings[category_mask].mean(axis=0)

print("Career profiles created:", len(career_embeddings))

Career profiles created: 43


In [ ]:
for category in career_embeddings:
    vector = career_embeddings[category]
    norm = np.linalg.norm(vector)

    if norm != 0:
        career_embeddings[category] = vector / norm

print("Career embeddings normalized.")

Career embeddings normalized.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

career_matrix = np.vstack([
    career_embeddings[category]
    for category in categories
])

similarity_scores = cosine_similarity(
    test_embeddings,
    career_matrix
)

print("Similarity matrix shape:", similarity_scores.shape)

Similarity matrix shape: (2384, 43)


In [ ]:
top_k = 5

top_k_results = []

for i in range(len(X_test)):

    scores = similarity_scores[i]

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, index in enumerate(top_indices, start=1):
        results.append({
            "rank": rank,
            "career": categories[index],
            "similarity": float(scores[index])
        })

    top_k_results.append(results)

print("Top-5 career ranking generated.")

Top-5 career ranking generated.


In [ ]:
sample_index = 0

print("Actual Career:")
print(y_test.iloc[sample_index])

print("\nTop-5 Predicted Careers:")

for result in top_k_results[sample_index]:
    print(
        f"{result['rank']}. "
        f"{result['career']} "
        f"({result['similarity']:.4f})"
    )

Actual Career:
Public Relations

Top-5 Predicted Careers:
1. Advocate (0.7758)
2. Health and Fitness (0.7634)
3. Testing (0.7186)
4. Education (0.7113)
5. Data Science (0.7110)


In [ ]:
actual_labels = y_test.tolist()

top1_correct = 0
top3_correct = 0
top5_correct = 0

for i, actual in enumerate(actual_labels):

    predicted_careers = [
        result["career"]
        for result in top_k_results[i]
    ]

    if actual in predicted_careers[:1]:
        top1_correct += 1

    if actual in predicted_careers[:3]:
        top3_correct += 1

    if actual in predicted_careers[:5]:
        top5_correct += 1

top1_accuracy = top1_correct / len(actual_labels)
top3_accuracy = top3_correct / len(actual_labels)
top5_accuracy = top5_correct / len(actual_labels)

print("==============================")
print("SENTENCE-BERT TOP-K RESULTS")
print("==============================")

print(f"Top-1 Accuracy: {top1_accuracy * 100:.2f}%")
print(f"Top-3 Accuracy: {top3_accuracy * 100:.2f}%")
print(f"Top-5 Accuracy: {top5_accuracy * 100:.2f}%")

SENTENCE-BERT TOP-K RESULTS
Top-1 Accuracy: 64.77%
Top-3 Accuracy: 81.50%
Top-5 Accuracy: 87.12%


In [ ]:
import numpy as np

# Convert similarity scores into confidence scores
temperature = 0.1

exp_scores = np.exp(
    (similarity_scores - np.max(similarity_scores, axis=1, keepdims=True))
    / temperature
)

confidence_scores = exp_scores / exp_scores.sum(
    axis=1,
    keepdims=True
)

print("Confidence scores generated.")

Confidence scores generated.


In [ ]:
sample_index = 0

print("Resume:", sample_index)
print("Actual Career:", y_test.iloc[sample_index])

print("\nTop-5 Career Predictions:")

top_indices = np.argsort(
    similarity_scores[sample_index]
)[::-1][:5]

for rank, index in enumerate(top_indices, 1):
    print(
        f"{rank}. {categories[index]} "
        f"- Confidence: {confidence_scores[sample_index][index] * 100:.2f}%"
    )

Resume: 0
Actual Career: Public Relations

Top-5 Career Predictions:
1. Advocate - Confidence: 8.83%
2. Health and Fitness - Confidence: 7.80%
3. Testing - Confidence: 4.99%
4. Education - Confidence: 4.63%
5. Data Science - Confidence: 4.62%


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

skill_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=3,
    max_features=5000
)

career_skill_profiles = {}

for category in categories:
    category_texts = X_train[y_train == category]

    profile_matrix = skill_vectorizer.fit_transform(
        category_texts
    )

    career_profile = np.asarray(
        profile_matrix.mean(axis=0)
    ).flatten()

    career_skill_profiles[category] = career_profile

print("Career skill profiles created:", len(career_skill_profiles))

Career skill profiles created: 43


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# One common vocabulary for all careers
skill_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=3,
    max_features=5000
)

# Fit ONLY on training resumes
skill_vectorizer.fit(X_train)

# Transform all training and test resumes
X_train_skill = skill_vectorizer.transform(X_train)
X_test_skill = skill_vectorizer.transform(X_test)

print("Training skill matrix:", X_train_skill.shape)
print("Testing skill matrix:", X_test_skill.shape)

Training skill matrix: (9532, 5000)
Testing skill matrix: (2384, 5000)


In [ ]:
career_skill_profiles = {}

for category in categories:
    mask = (y_train.values == category)

    # Use the SAME vocabulary for every category
    category_matrix = X_train_skill[mask]

    # Average TF-IDF vector for this career
    career_profile = np.asarray(
        category_matrix.mean(axis=0)
    ).flatten()

    career_skill_profiles[category] = career_profile

print(
    "Career skill profiles created:",
    len(career_skill_profiles)
)

Career skill profiles created: 43


In [ ]:
skill_alignment_scores = []

for i in range(len(X_test)):

    resume_vector = X_test_skill[i].toarray().flatten()

    scores = {}

    resume_norm = np.linalg.norm(resume_vector)

    for category in categories:

        career_vector = career_skill_profiles[category]

        career_norm = np.linalg.norm(career_vector)

        if resume_norm == 0 or career_norm == 0:
            score = 0.0
        else:
            score = np.dot(
                resume_vector,
                career_vector
            ) / (resume_norm * career_norm)

        scores[category] = score

    skill_alignment_scores.append(scores)

print("Skill alignment scores generated.")

Skill alignment scores generated.


In [ ]:
alignment_values = []

for i in range(len(X_test)):

    top_category = top_k_results[i][0]["career"]

    alignment = skill_alignment_scores[i][top_category]

    alignment_values.append(alignment)

alignment_values = np.array(alignment_values)

print(
    f"Average Skill Alignment: "
    f"{alignment_values.mean() * 100:.2f}%"
)

Average Skill Alignment: 44.32%


In [ ]:
sample_index = 0

print("======================================")
print("CAREERCAST - SAMPLE CAREER PREDICTION")
print("======================================")

print("\nActual Career:")
print(y_test.iloc[sample_index])

print("\nTop-5 Career Recommendations:")

top_indices = np.argsort(
    similarity_scores[sample_index]
)[::-1][:5]

for rank, index in enumerate(top_indices, 1):

    career = categories[index]

    confidence = (
        confidence_scores[sample_index][index] * 100
    )

    alignment = (
        skill_alignment_scores[sample_index][career] * 100
    )

    print(
        f"{rank}. {career}"
        f" | Confidence: {confidence:.2f}%"
        f" | Skill Alignment: {alignment:.2f}%"
    )

CAREERCAST - SAMPLE CAREER PREDICTION

Actual Career:
Public Relations

Top-5 Career Recommendations:
1. Advocate | Confidence: 8.83% | Skill Alignment: 30.57%
2. Health and Fitness | Confidence: 7.80% | Skill Alignment: 25.17%
3. Testing | Confidence: 4.99% | Skill Alignment: 26.41%
4. Education | Confidence: 4.63% | Skill Alignment: 25.01%
5. Data Science | Confidence: 4.62% | Skill Alignment: 25.11%
